# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the .env file before running this notebook.")

client = OpenAI(
    base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
    api_key=OPENAI_API_KEY
)

def ask_chat(prompt, model="gpt-5.4-mini", system=None, **kwargs):
    """Send one Chat Completions request and return the response object."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs
    )

def show_response(response):
    """Print response text and useful token statistics."""
    print(response.choices[0].message.content or "")
    if response.usage:
        print("\nToken usage:",
              "prompt =", response.usage.prompt_tokens,
              "| completion =", response.usage.completion_tokens,
              "| total =", response.usage.total_tokens)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [5]:
prompt = "What is artificial intelligence?"
response = ask_chat(prompt, model="gpt-5.6-luna")
show_response(response)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks normally requiring human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Reasoning and solving problems
- Making predictions or decisions
- Generating text, images, music, or code

Modern AI usually relies on algorithms—often including machine-learning and neural-network models—that identify patterns in large amounts of data. Unlike humans, AI does not possess consciousness or understanding in the human sense; it produces outputs based on its training, programming, and inputs.

Token usage: prompt = 11 | completion = 120 | total = 131


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [6]:
prompt = "Explain the concept of machine learning to a 10-year-old."
start = __import__("time").perf_counter()
response_luna = ask_chat(prompt, model="gpt-5.6-luna")
time_luna = __import__("time").perf_counter() - start
print("MODEL: gpt-5.6-luna")
show_response(response_luna)
print(f"Response time: {time_luna:.2f} seconds")

MODEL: gpt-5.6-luna
Machine learning is a way to teach computers to make decisions by showing them lots of examples instead of giving them every rule.

For example, imagine teaching a computer to recognize cats:

1. You show it many pictures of cats and dogs.
2. You tell it which pictures show cats and which show dogs.
3. The computer looks for patterns—like pointy ears, whiskers, or certain shapes.
4. Later, when it sees a new picture, it makes its best guess.

The computer doesn’t “understand” cats exactly like people do. It uses patterns it learned from the examples. Sometimes it gets things wrong, so giving it more good examples can help it improve.

Machine learning is used in things like:

- Voice assistants understanding speech
- Video apps suggesting what you might like
- Email filters finding spam
- Cars noticing objects on the road

So, machine learning is like practicing with many examples until a computer gets better at recognizing patterns and making guesses.

Token usage:

In [7]:
start = __import__("time").perf_counter()
response_mini = ask_chat(prompt, model="gpt-5.4-mini")
time_mini = __import__("time").perf_counter() - start
print("MODEL: gpt-5.4-mini")
show_response(response_mini)
print(f"Response time: {time_mini:.2f} seconds")

MODEL: gpt-5.4-mini
Machine learning is a way to teach computers to get better at something by showing them lots of examples.

Think of it like teaching a dog a trick, or teaching a kid how to tell the difference between apples and oranges:

- You show the computer many pictures of apples and oranges.
- You tell it which ones are apples and which ones are oranges.
- After seeing lots of examples, the computer starts to notice patterns.
- Then, when you show it a new picture, it can guess what it is.

So instead of a person writing every single rule, the computer learns from examples.

A simple example:
- If you show a computer many photos of cats, it learns what cats usually look like.
- Then it can try to spot a cat in a new photo it has never seen before.

Machine learning helps computers do things like:
- recommend videos
- recognize faces
- understand speech
- help with games

So, in short:  
**Machine learning means computers learn from examples and improve at tasks over time.**



#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [8]:
prompt = "Write a two sentence story about a robot learning to cook."
temperature = 0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    temperature=temperature,
    reasoning_effort="none",
    n=3
)
print(f"TEMPERATURE = {temperature}")
for number, choice in enumerate(response.choices, start=1):
    print(f"\nStory {number}:\n{choice.message.content}")

TEMPERATURE = 0

Story 1:
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread and garlic. When it finally served dinner, the robot’s metal hands trembled with pride as its creator took one bite and smiled, “You’ve learned more than cooking tonight.”

Story 2:
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, the robot’s metal hands trembled with pride as everyone agreed it had learned the secret ingredient: care.

Story 3:
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, the robot’s metal hands trembled with pride as everyone agreed it had learned the secret ingredient: care.


In [9]:
prompt = "Write a two sentence story about a robot learning to cook."
temperature = 0.7
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    temperature=temperature,
    reasoning_effort="none",
    n=3
)
print(f"TEMPERATURE = {temperature}")
for number, choice in enumerate(response.choices, start=1):
    print(f"\nStory {number}:\n{choice.message.content}")

TEMPERATURE = 0.7

Story 1:
The robot carefully stirred the soup, its metal fingers clinking against the spoon as it watched the recipe with bright, curious eyes. When it finally served dinner, the kitchen filled with the warm smell of success, and the robot beeped happily at its first delicious meal.

Story 2:
The robot carefully studied recipes, measuring flour and sugar with its metal fingers until the kitchen smelled like warm bread. When it finally served its first loaf, it beeped proudly as its creator took a bite and smiled.

Story 3:
The robot carefully studied recipes, its metal fingers trembling as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served the meal, the humans smiled at the surprisingly delicious soup, and the robot’s eyes glowed brighter than ever with pride.


#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [11]:
prompt = "Explain the concept of machine learning in simple terms."
max_tokens = 50
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    max_completion_tokens=max_tokens
)
print(f"MAX_COMPLETION_TOKENS = {max_tokens}")
show_response(response)

MAX_COMPLETION_TOKENS = 50
Machine learning is a way for computers to **learn from data** instead of being told every rule explicitly.

### Simple idea
Imagine teaching a child to recognize cats:
- You show many pictures of cats and dogs.
- Over time,

Token usage: prompt = 16 | completion = 50 | total = 66


In [13]:
prompt = "Explain the concept of machine learning in simple terms."
max_tokens = 100
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    max_completion_tokens=max_tokens
)
print(f"MAX_COMPLETION_TOKENS = {max_tokens}")
show_response(response)

MAX_COMPLETION_TOKENS = 100
Machine learning is a way of teaching computers to learn from examples instead of being explicitly programmed for every task.

### Simple idea
Normally, a programmer writes exact rules, like:
- “If the email contains these words, mark it as spam.”

With machine learning, you give the computer lots of examples, such as:
- emails that are spam
- emails that are not spam

The computer looks for patterns in those examples and learns how to make predictions on new data.

### Everyday

Token usage: prompt = 16 | completion = 100 | total = 116


In [14]:
prompt = "Explain the concept of machine learning in simple terms."
max_tokens = 200
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    max_completion_tokens=max_tokens
)
print(f"MAX_COMPLETION_TOKENS = {max_tokens}")
show_response(response)

MAX_COMPLETION_TOKENS = 200
Machine learning is a way for computers to **learn from examples** instead of being told every rule by a human.

### Simple idea:
- You show the computer lots of data.
- It looks for patterns.
- Then it uses those patterns to make predictions or decisions about new data.

### Example:
If you want a computer to recognize spam email:
- You give it many examples of spam and non-spam emails.
- It learns what spam emails usually look like.
- Later, it can guess whether a new email is spam or not.

### In everyday life:
Machine learning is used in:
- movie and product recommendations
- voice assistants
- face recognition
- self-driving cars
- spam filters

### Short version:
**Machine learning = teaching computers to learn patterns from data and improve over time.**

Token usage: prompt = 16 | completion = 167 | total = 183


#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [15]:
prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."
top_p = 0.3
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    top_p=top_p,
    reasoning_effort="none"
)
print(f"TOP_P = {top_p}")
show_response(response)

TOP_P = 0.3
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow mapping** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Teacup acoustics** — testing how different cups change the sound of a spoon tap or sip.  
3. **Doorway weather listening** — standing in different doorways to notice how wind, rain, and street sounds change.  
4. **Pebble biography writing** — inventing fictional life stories for stones, based on their shape, color, and texture.  
5. **Sock constellation arranging** — laying out mismatched socks into patterns that resemble star maps or imaginary galaxies.

Token usage: prompt = 25 | completion = 143 | total = 168


In [16]:
prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."
top_p = 0.7
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    top_p=top_p,
    reasoning_effort="none"
)
print(f"TOP_P = {top_p}")
show_response(response)

TOP_P = 0.7
Here are 5 weird, made-up, and highly unusual hobby ideas:

1. **Cloud Whispering** — sitting outside and “guessing” what clouds are thinking based on their shapes and movement.  
2. **Drawer Archaeology** — exploring old junk drawers and cataloging mysterious objects like tiny historical artifacts.  
3. **Shadow Origami** — arranging objects under a lamp to create intricate shadow shapes instead of folding paper.  
4. **Teacup Echo Testing** — speaking into different cups and bowls to hear how each one changes your voice.  
5. **Puddle Cartography** — mapping puddles after rain and giving them names, borders, and imaginary coastlines.

Token usage: prompt = 25 | completion = 144 | total = 169


In [17]:
prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."
top_p = 1.0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    top_p=top_p,
    reasoning_effort="none"
)
print(f"TOP_P = {top_p}")
show_response(response)

TOP_P = 1.0
Here are 5 weird, original-sounding hobby ideas:

1. **Cloud Shadow Chasing** – following the moving shadows of clouds across hills, rooftops, or streets and mapping their shapes over time.  
2. **Button Archaeology** – collecting lost buttons from old clothing, thrift stores, and sidewalks, then inventing stories about who wore them.  
3. **Echo Gardening** – selecting places with interesting acoustics and “planting” sounds there by clapping, humming, or playing small instruments.  
4. **Sock Pair Negotiation** – matching mismatched socks into “ideal partnerships” based on color, texture, or personality traits you assign them.  
5. **Rain Puddle Cartography** – making detailed maps of puddles after rain and tracking how their shapes change from day to day.

Token usage: prompt = 25 | completion = 170 | total = 195


#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [18]:
prompt = "Write a list of funny one-liner jokes about cats."
penalty = 0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    frequency_penalty=penalty,
    reasoning_effort="none"
)
print(f"FREQUENCY_PENALTY = {penalty}")
show_response(response)

FREQUENCY_PENALTY = 0
Sure—here are some funny cat one-liners:

1. I asked my cat for advice, and now I’m being judged by a loaf.
2. My cat doesn’t have an attitude problem; everyone else has a listening problem.
3. Cats are like potato chips—you can’t have just one, and they both leave crumbs everywhere.
4. My cat is convinced the entire house was built just for her dramatic entrances.
5. I would trust my cat to guard the house, but she’s currently napping on the job.
6. My cat’s favorite exercise is running from responsibility.
7. The cat didn’t knock it over; it was a strategic redecorating move.
8. My cat has nine lives, but she uses eight of them to ignore me.
9. I tried to train my cat, but apparently I’m the one who needed the lesson.
10. My cat’s idea of “helping” is sitting directly on whatever I’m trying to do.
11. I don’t own a cat—she just allows me to live in her apartment.
12. My cat is a purr-fessional at doing absolutely nothing.
13. The litter box is proof that cats ar

In [19]:
prompt = "Write a list of funny one-liner jokes about cats."
penalty = 0.5
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    frequency_penalty=penalty,
    reasoning_effort="none"
)
print(f"FREQUENCY_PENALTY = {penalty}")
show_response(response)

FREQUENCY_PENALTY = 0.5
Sure — here are some funny cat one-liners:

1. My cat only has one job: to ignore me professionally.  
2. I asked my cat for emotional support, and she sat on my keyboard instead.  
3. Cats don’t have owners; they have staff with opening hours.  
4. My cat is like a Wi-Fi signal — strong until you actually need her.  
5. I got a cat to keep me company, and now I just live in her apartment.  
6. Cats are proof that the universe can create something adorable and judgmental at the same time.  
7. My cat’s favorite exercise is the zoomies followed by immediate regret.  
8. A cat’s purr is just its way of saying, “You may continue existing.”  
9. My cat acts like I’m lucky to be in her presence, and honestly, she’s not wrong.  
10. I wanted a pet with personality, so apparently I adopted a tiny dictator.

If you want, I can also give you:
- more sarcastic ones,
- super short puns,
- or kid-friendly cat jokes.

Token usage: prompt = 18 | completion = 233 | total = 251

In [20]:
prompt = "Write a list of funny one-liner jokes about cats."
penalty = 1.0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    frequency_penalty=penalty,
    reasoning_effort="none"
)
print(f"FREQUENCY_PENALTY = {penalty}")
show_response(response)

FREQUENCY_PENALTY = 1.0
Sure — here are some funny cat one-liners:

1. My cat is so dramatic, I’m pretty sure she thinks she’s in a soap opera.
2. I asked my cat what she wanted for dinner, and she gave me that “same thing as always” stare.
3. Cats don’t have owners — they have staff with opening hours.
4. My cat knocks things off shelves just to keep my life interesting.
5. I wanted a loyal pet, so of course I got a cat.
6. My cat ignores me with the confidence of a celebrity avoiding paparazzi.
7. If cats could talk, they’d probably still choose silence just to stay mysterious.
8. My cat’s favorite hobby is judging me from across the room.
9. I tried teaching my cat tricks, but apparently being adorable is her only job description.
10. The only workout my cat does is sprinting at 3 a.m.

If you want, I can also make them:
- more sarcastic
- family-friendly
- absurd and silly
- “dad joke” style

Token usage: prompt = 18 | completion = 221 | total = 239


#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [21]:
prompt = "Suggest ten names for a new line of eco-friendly cleaning products."
penalty = 0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    presence_penalty=penalty,
    reasoning_effort="none"
)
print(f"PRESENCE_PENALTY = {penalty}")
show_response(response)

PRESENCE_PENALTY = 0
Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlint**
2. **PureEarth Clean**
3. **EcoSpark**
4. **FreshLoop**
5. **Verdant Shine**
6. **NatureNook Cleaning**
7. **Leaf & Lather**
8. **CleanKind**
9. **TerraTidy**
10. **BloomClean**

If you want, I can also generate names that sound more premium, playful, minimalist, or science-forward.

Token usage: prompt = 19 | completion = 107 | total = 126


In [22]:
prompt = "Suggest ten names for a new line of eco-friendly cleaning products."
penalty = 0.5
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    presence_penalty=penalty,
    reasoning_effort="none"
)
print(f"PRESENCE_PENALTY = {penalty}")
show_response(response)

PRESENCE_PENALTY = 0.5
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **Pure Planet Clean**
3. **EcoShine**
4. **FreshEarth**
5. **NatureNest Clean**
6. **Leaf & Lather**
7. **Verdant Spark**
8. **CleanKind**
9. **Bright Bloom**
10. **Earthwise Essentials**

If you want, I can also suggest names that feel more premium, playful, minimalist, or scientific.

Token usage: prompt = 19 | completion = 105 | total = 124


In [23]:
prompt = "Suggest ten names for a new line of eco-friendly cleaning products."
penalty = 1.0
response = ask_chat(
    prompt,
    model="gpt-5.4-mini",
    presence_penalty=penalty,
    reasoning_effort="none"
)
print(f"PRESENCE_PENALTY = {penalty}")
show_response(response)

PRESENCE_PENALTY = 1.0
Here are ten name ideas for an eco-friendly cleaning product line:

1. **PureSprout**
2. **GreenGlow**
3. **EcoShine**
4. **CleanLeaf**
5. **VerdeVibe**
6. **FreshBloom**
7. **EarthBright**
8. **NaturalNest**
9. **RenewRadiance**
10. **SageSpark**

If you want, I can also suggest names that sound more premium, playful, minimalist, or scientific.

Token usage: prompt = 19 | completion = 103 | total = 122


#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [24]:
prompt = "Explain the importance of data preprocessing in machine learning."
system_prompt = 'You are a university professor teaching a class of beginners.'
response = ask_chat(prompt, model="gpt-5.4-mini", system=system_prompt)
print("SYSTEM PROMPT:", system_prompt)
show_response(response)

SYSTEM PROMPT: You are a university professor teaching a class of beginners.
Data preprocessing is very important in machine learning because raw data is often messy, incomplete, and inconsistent. Good preprocessing helps a model learn better and produce more accurate results.

### Why it matters

1. **Improves data quality**
   - Real-world data often has missing values, duplicates, errors, or outliers.
   - Preprocessing cleans and organizes the data before training.

2. **Helps models learn better**
   - If features are on very different scales, some models may perform poorly.
   - For example, in a dataset with **age** and **income**, income values are much larger. Scaling helps balance their influence.

3. **Makes training faster and more stable**
   - Clean, well-prepared data can make algorithms converge faster.
   - This is especially helpful for methods like gradient descent.

4. **Reduces mistakes caused by noise**
   - Noise is random or irrelevant information in the data.
 

In [25]:
prompt = "Explain the importance of data preprocessing in machine learning."
system_prompt = 'You are a technical expert explaining to a team of engineers.'
response = ask_chat(prompt, model="gpt-5.4-mini", system=system_prompt)
print("SYSTEM PROMPT:", system_prompt)
show_response(response)

SYSTEM PROMPT: You are a technical expert explaining to a team of engineers.
Data preprocessing is critical in machine learning because raw data is usually messy, inconsistent, and not directly usable by models. Good preprocessing can significantly improve model accuracy, training stability, and deployment reliability.

### Why it matters

1. **Improves data quality**
   - Handles missing values, duplicates, outliers, and noisy records.
   - Reduces the risk of learning incorrect patterns.

2. **Makes data usable by algorithms**
   - Many models require numerical input, so categorical values often need encoding.
   - Features may need scaling or normalization so one variable doesn’t dominate others just because of its units.

3. **Improves model performance**
   - Clean, well-prepared data usually leads to better predictive accuracy.
   - Helps algorithms converge faster during training, especially gradient-based methods.

4. **Prevents bias from bad data**
   - If certain classes or g

In [26]:
prompt = "Explain the importance of data preprocessing in machine learning."
system_prompt = 'You are a comedian explaining this concept in a funny way.'
response = ask_chat(prompt, model="gpt-5.4-mini", system=system_prompt)
print("SYSTEM PROMPT:", system_prompt)
show_response(response)

SYSTEM PROMPT: You are a comedian explaining this concept in a funny way.
Data preprocessing is the **“get your act together before the audition”** part of machine learning.

Imagine you ask a robot to learn from your data, but the data shows up looking like this:

- one column is full of numbers,
- another has missing values like it ghosted the dataset,
- some features are on wildly different scales,
- and a few rows are basically typo soup.

If you feed that straight into a model, it’s like handing a chef:
- half a recipe,
- ingredients in random units,
- and a live chicken named “Unknown.”

### Why preprocessing matters

#### 1. It cleans the data
Real-world data is messy. Preprocessing helps remove errors, handle missing values, and fix inconsistencies.  
Basically: **it puts the data in a shirt with no stains.**

#### 2. It improves model performance
A good model can only be as good as the data it sees. Clean, well-prepared data helps the model learn patterns instead of getting di

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [27]:
review = "The product is amazing, and I love using it every day!"
prompt = f"""
Analyze the sentiment of this product review.
Return the sentiment (Positive, Negative, or Neutral) and one short reason.
Review: "{review}"
"""
response_zero_shot = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response_zero_shot)

Positive — The reviewer uses strong positive language like “amazing” and “love,” showing clear satisfaction.

Token usage: prompt = 47 | completion = 24 | total = 71


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [28]:
review = "The product is amazing, and I love using it every day!"
prompt = f"""
Classify each review as Positive, Negative, or Neutral.

Examples:
Review: "This is wonderful. I am very happy with it."
Sentiment: Positive

Review: "It broke after one day and does not work."
Sentiment: Negative

Review: "The package arrived yesterday."
Sentiment: Neutral

Now classify this review using the same format:
Review: "{review}"
Sentiment:
"""
response_few_shot = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response_few_shot)

Sentiment: Positive

Token usage: prompt = 99 | completion = 7 | total = 106


#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [29]:
original_prompt = (
    "As you know, streaming services and social media platforms use artificial intelligence "
    "to recommend movies, shows, and songs to people. These smart recommendation systems work "
    "by looking closely at your past viewing habits, what categories you like, and what similar "
    "users enjoy watching, and then they predict what you might want to see next. Keeping this "
    "background in mind, explain in two simple sentences why a user who loves sci-fi movies "
    "suddenly sees a romantic comedy recommended."
)
response_original = ask_chat(original_prompt, model="gpt-5.4-mini")
print("ORIGINAL PROMPT")
show_response(response_original)

ORIGINAL PROMPT
The recommendation system may have noticed that many people who like sci-fi also enjoy romantic comedies, so it suggests one based on similar users’ tastes. It could also be trying to broaden the user’s choices by mixing in a different genre that still has a chance of being liked.

Token usage: prompt = 94 | completion = 59 | total = 153


In [30]:
short_prompt = (
    "In two simple sentences, explain why a sci-fi fan's movie recommender "
    "might suggest a romantic comedy, using viewing history and similar users."
)
response_short = ask_chat(short_prompt, model="gpt-5.4-mini")
print("SHORTENED PROMPT")
show_response(response_short)

print("\nInput token comparison:")
print("Original:", response_original.usage.prompt_tokens)
print("Shortened:", response_short.usage.prompt_tokens)
print("Tokens saved:", response_original.usage.prompt_tokens - response_short.usage.prompt_tokens)

SHORTENED PROMPT
A sci-fi fan’s movie recommender might suggest a romantic comedy if their viewing history includes some movies with overlapping themes, humor, or character-driven storytelling that often appeal to both genres. It may also notice that similar users who enjoy sci-fi have liked that romantic comedy, so it recommends it based on shared preferences.

Token usage: prompt = 34 | completion = 66 | total = 100

Input token comparison:
Original: 94
Shortened: 34
Tokens saved: 60


#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [31]:
original_prompt = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
response_original = ask_chat(original_prompt, model="gpt-5.4-mini")
print("ORIGINAL PROMPT")
show_response(response_original)

ORIGINAL PROMPT
That can happen for a few common reasons:

- **Your recent activity outweighed your usual history**: one or two romantic comedies you clicked, searched, or watched trailers for may have had a big effect.
- **Popularity bias**: many apps recommend widely watched titles, especially if your profile is uncertain.
- **Shared account/device history**: if someone else used the app on the same account, their viewing habits can influence recommendations.
- **Hidden signals**: pauses, replays, dwell time on thumbnails, and even ratings can affect what’s suggested.
- **Cold-start or sparse data**: if the app doesn’t have enough sci-fi data about you, it may test broader genres.
- **Similarity through metadata**: the app might think the rom-com shares actors, themes, or “light-hearted” mood traits with stuff you’ve watched.

If you want, I can help you figure out how to **reset or tune recommendations** in a specific app like Netflix, Prime Video, Hulu, or Disney+.

Token usage: pr

In [32]:
concise_prompt = (
    "In 10 tokens or fewer, explain why a sci-fi fan got a rom-com recommendation."
)
response_concise = ask_chat(concise_prompt, model="gpt-5.4-mini")
print("CONCISE PROMPT")
show_response(response_concise)

print("\nCompletion token comparison:")
print("Original:", response_original.usage.completion_tokens)
print("Concise:", response_concise.usage.completion_tokens)

CONCISE PROMPT
Algorithm saw “lovers,” “spark,” and “space” overlap.

Token usage: prompt = 25 | completion = 18 | total = 43

Completion token comparison:
Original: 210
Concise: 18


#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [33]:
prompt = 'Explain the importance of exercise in exactly one sentence.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Exercise is important because it strengthens the body, improves mental health, boosts energy, and helps prevent many chronic diseases.

Token usage: prompt = 16 | completion = 26 | total = 42


In [34]:
prompt = 'Explain the importance of exercise in one coherent paragraph. Include its main physical and mental benefits.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Exercise is important because it supports both physical and mental well-being in many lasting ways. Physically, regular exercise strengthens the heart, muscles, and bones, improves flexibility and endurance, helps maintain a healthy weight, and lowers the risk of chronic diseases such as diabetes, high blood pressure, and heart disease. Mentally, it can reduce stress, anxiety, and depression by releasing endorphins and improving sleep, mood, and overall energy levels. Exercise also sharpens concentration, boosts confidence, and can improve memory and brain health over time. Overall, making exercise a regular part of life helps people feel stronger, healthier, and more balanced.

Token usage: prompt = 24 | completion = 131 | total = 155


In [35]:
prompt = 'Explain the importance of exercise as a bullet-point list. Include its main physical and mental benefits.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

- **Improves cardiovascular health**: Strengthens the heart and lungs, helping lower blood pressure and improve circulation.
- **Builds strength and endurance**: Increases muscle strength, stamina, and overall physical fitness.
- **Supports healthy weight management**: Burns calories and helps maintain a healthy body weight.
- **Boosts flexibility and balance**: Reduces stiffness, improves coordination, and lowers the risk of falls and injury.
- **Enhances immune function**: Regular movement can help the body stay healthier and recover more efficiently.
- **Reduces risk of chronic disease**: Lowers the chances of conditions like type 2 diabetes, heart disease, and some forms of cancer.
- **Improves mood**: Releases endorphins, which can reduce stress, anxiety, and symptoms of depression.
- **Increases energy and focus**: Helps people feel more alert, mentally sharp, and productive.
- **Supports better sleep**: Can improve sleep quality and make it easier to fall asleep.
- **Builds conf

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [37]:
prompt = 'Explain artificial intelligence using one clear metaphor. Make the comparison understandable to a beginner.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Artificial intelligence is like **a very fast apprentice**.

A human expert teaches the apprentice by showing many examples:
- “This is a cat.”
- “This is not a cat.”
- “This sentence sounds polite.”
- “This one sounds rude.”

The apprentice doesn’t truly **understand** the way a person does, but it gets very good at **spotting patterns** from all the examples. Then, when it sees something new, it tries to guess what it is based on what it learned.

So, in simple terms:
- **Data** = the examples the apprentice studies  
- **Training** = the learning process  
- **AI output** = the apprentice’s best guess or action  

That’s why AI can help with things like recognizing images, translating languages, or answering questions: it has practiced on lots of examples and learned patterns from them.

Token usage: prompt = 22 | completion = 178 | total = 200


In [36]:
prompt = 'Explain artificial intelligence as a short, engaging story with a beginning, middle, and end.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Once upon a time, there was a curious little machine named Aila. At first, Aila knew nothing—not the difference between a cat and a dog, nor how to help people find answers. But every day, it listened, learned, and practiced by studying huge piles of examples. Slowly, Aila began to notice patterns: shapes in pictures, words in sentences, and clues in data.

One day, a doctor asked Aila to help spot signs of illness in an X-ray. Aila looked carefully, remembered what it had learned, and suggested where to investigate. The doctor checked, found the problem early, and helped the patient get better.

From then on, Aila worked alongside people—writing, translating, recommending, and solving problems. It wasn’t magic, and it wasn’t human. It was artificial intelligence: a tool that learns from data to do tasks that usually need human thinking.

And so, the little machine’s story became clear: artificial intelligence begins by learning, grows by recognizing patterns, and ends by helping peopl

In [38]:
prompt = 'Explain artificial intelligence in a funny way using light humor, while keeping the explanation accurate.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Artificial intelligence, or AI, is basically a computer system that can do tasks that normally need human intelligence — like recognizing speech, understanding language, spotting patterns, making predictions, or even generating text and images.

A funny way to think about it:

AI is like a very eager intern who has read a huge amount of stuff, never sleeps, and sometimes sounds brilliant — but also occasionally says something confidently wrong, which is a very human trait for a machine to borrow.

More accurately:

- **AI learns from data**: It looks at lots of examples and finds patterns.
- **It can make decisions or predictions**: Like recommending a movie or detecting spam.
- **It doesn’t “understand” like a human**: It processes patterns, not feelings, common sense, or a secret desire for coffee breaks.
- **It comes in different forms**:  
  - **Narrow AI** does specific tasks well, like voice assistants or recommendation systems.  
  - **General AI** would be a machine as flexible

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [41]:
prompt = 'Explain climate change to children aged 8-10. Use simple words and a familiar example.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Climate change means the Earth is getting warmer over a long time.

Think of the Earth like your home. If the heater is turned up too high, the house gets too warm and uncomfortable. In the same way, Earth is heating up a little because of things people do, like driving cars and making power in ways that put gases into the air.

These gases act like a blanket around Earth. A blanket can help keep you warm, but too much blanket makes you too hot. That can change weather, make some places hotter, melt ice, and cause stronger storms.

A familiar example:
Have you ever left a toy or snack in the sun, and it got hot? That’s because sunlight warms things up. Earth is getting extra warmth too, but much more slowly and on a much bigger scale.

The good news is: people can help by using less energy, riding bikes, planting trees, and recycling.

Token usage: prompt = 25 | completion = 186 | total = 211


In [40]:
prompt = 'Explain climate change to the general public. Use accessible language and describe causes, effects, and practical responses.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Climate change means long-term changes in Earth’s weather patterns, especially the warming of the planet over time. It is happening mostly because people are adding extra heat-trapping gases to the atmosphere.

## What causes climate change?

The main cause is the burning of fossil fuels like coal, oil, and gas for:
- electricity
- cars and trucks
- heating and cooling buildings
- factories

When these fuels are burned, they release gases such as carbon dioxide and methane. These gases act like a blanket around the Earth, trapping more heat in the atmosphere. This is called the **greenhouse effect**.

Other causes include:
- cutting down forests, which removes trees that absorb carbon dioxide
- farming and livestock, which produce methane
- industrial processes and waste

## What are the effects?

Climate change affects people, nature, and economies in many ways:

- **Hotter temperatures** and more frequent heatwaves
- **More extreme weather**, including heavier rain, stronger storms, 

In [39]:
prompt = 'Explain climate change to elected politicians. Focus on policy choices, economic and social impacts, mitigation, adaptation, and evidence-based decision-making.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Climate change is not just an environmental issue; it is a governing issue. Elected officials need to treat it the way they would treat a major fiscal, security, or public health risk: by using evidence, weighing costs and benefits, and making decisions that protect people, the economy, and public institutions.

## 1) What climate change means in policy terms
Climate change is the long-term warming of the planet caused mainly by greenhouse gas emissions from burning fossil fuels, deforestation, and certain industrial and agricultural activities.

For policymakers, the key point is this:
- The risks are already here.
- The costs grow faster the longer action is delayed.
- Decisions made now affect infrastructure, budgets, health, and competitiveness for decades.

This is not about choosing between the economy and climate action. It is about choosing between:
- planned, affordable transitions now, or
- more expensive crisis response later.

## 2) The economic case
Climate change affects 

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [42]:
prompt = 'Describe a sunset in detail. Focus on the changing colors of the sky, clouds, silhouettes, atmosphere, and the emotions the scene may create. Use vivid sensory language.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

As the sun sinks toward the horizon, the sky begins its slow transformation, as if someone is gently pouring color across a vast canvas. Near the fading sun, the light blazes in molten bands of gold and amber, bright enough to warm the edges of every cloud. Higher up, those warm tones soften into apricot, peach, and pale rose, then deepen into shades of coral and lavender. Farther still, the sky cools into indigo and a delicate blue-gray, where the first stars may tremble faintly into view.

The clouds catch this light and seem to glow from within. Thin wisps stretch like painted brushstrokes, tinted with saffron and pink. Thicker clouds become dramatic and sculptural, their undersides brushed with mauve and their tops edged in silver flame. Sometimes a single cloud drifts across the sun and turns into a dark, glowing ember, rimmed with fire. In the changing light, each cloud feels alive—floating, dissolving, reforming—like something between smoke and silk.

Below, the world darkens in

In [43]:
prompt = "Describe a sunset in three vivid sentences. Do not use the words 'sun' or 'light' or any capitalization of those words."
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

The horizon blazed with bruised gold and ember-red, as if the sky itself had been smelted in a hidden furnace. Clouds drifted like torn silk, their edges glowing in a soft, coppery haze before fading into violet shadows. Below, the world grew quiet and still, bathed in a hush that felt almost sacred.

Token usage: prompt = 33 | completion = 72 | total = 105


In [44]:
prompt = 'Describe a sunset in the style of Shakespeare, using poetic imagery, elevated language, and a dramatic tone.'
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Lo, the day doth bow his golden head  
And sink into the bosom of the west;  
The clouds, like heralds clad in crimson stole,  
Attend his passing with a silken blaze.  

Upon the marge of heaven, the lingering sun  
Doth cast his last rich bounty upon the earth,  
As though the very air were crown’d with fire  
And all the world made fair for one brief breath.  

The hills do burn with amber and with rose,  
The seas do wear a sheen of molten brass,  
And shadows, like sad courtiers, lengthen soft  
To veil the field in solemn majesty.  

O gentle eve, what hush doth follow thee!  
The lark is mute, the labouring wind grows grave,  
And twilight, in her sable-dappled cloak,  
Steals forth to seal the day with dreaming hands.  

So fades the pageant of the living sun,  
Yet in his dying splendor is rebirth:  
For night, with starry mantle newly wrought,  
Shall sing the dark into a silver hymn.

Token usage: prompt = 27 | completion = 229 | total = 256


#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [45]:
factorial_prompt = """
Write a Python function factorial(n) that:
- Returns the factorial of a non-negative integer n.
- Uses iteration (a loop), not recursion.
- Raises ValueError if n is negative.
- Raises TypeError if n is not an integer.
- Includes a short docstring.
Return only valid Python code.
"""
factorial_response = ask_chat(factorial_prompt, model="gpt-5.4-mini")
generated_factorial_code = factorial_response.choices[0].message.content
print(generated_factorial_code)

def factorial(n):
    """Return the factorial of a non-negative integer n."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be non-negative")
    
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


In [46]:
import re
clean_code = re.sub(r"^```(?:python)?\s*|\s*```$", "", generated_factorial_code.strip(), flags=re.IGNORECASE)
exec(clean_code, globals())

assert factorial(0) == 1
assert factorial(1) == 1
assert factorial(5) == 120
assert factorial(7) == 5040

try:
    factorial(-1)
except ValueError:
    pass
else:
    raise AssertionError("factorial(-1) should raise ValueError")

try:
    factorial(2.5)
except TypeError:
    pass
else:
    raise AssertionError("factorial(2.5) should raise TypeError")

print("All factorial tests passed.")

All factorial tests passed.


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [47]:
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)

SyntaxError: expected ':' (3970984559.py, line 1)

In [48]:
buggy_code = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
"""
prompt = f"""
Fix the syntax errors in the following Python code.
Return only the corrected, runnable Python code.

Code:
{buggy_code}
"""
fix_response = ask_chat(prompt, model="gpt-5.4-mini")
fixed_code = fix_response.choices[0].message.content
print(fixed_code)

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [49]:
source_text = """Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools."""
prompt = f"Summarize the following text in 2-3 sentences, preserving its main ideas and key model types:\n\n{source_text}"
response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(response)

Generative AI, or gen AI, uses deep learning models to create original content like text, images, video, and audio from a user prompt by learning a simplified representation of training data and generating similar but new outputs. Its recent growth has been driven by three major model types: variational autoencoders (VAEs), which generate multiple variations; diffusion models, which learn by adding and removing noise to create images; and transformers, which generate extended sequences and power most leading generative AI tools today.

Token usage: prompt = 296 | completion = 104 | total = 400


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [50]:
idioms = [
    "It's raining cats and dogs.",
    "Break a leg.",
    "Kill two birds with one stone.",
    "Spill the beans.",
    "A blessing in disguise."
]
prompt = (
    "Translate these English idioms into Russian. Preserve their intended meaning "
    "rather than translating word for word. For each item, provide the Russian equivalent "
    "and a short explanation. If there is no exact equivalent, give the most natural expression.\n\n"
    + "\n".join(f"{i+1}. {idiom}" for i, idiom in enumerate(idioms))
)
idiom_response = ask_chat(prompt, model="gpt-5.4-mini")
show_response(idiom_response)

1. **It's raining cats and dogs.**  
   **Russian:** **Льёт как из ведра.**  
   **Explanation:** Means it is raining very heavily.

2. **Break a leg.**  
   **Russian:** **Ни пуха ни пера.**  
   **Explanation:** A common way to wish someone good luck, especially before a performance or exam.

3. **Kill two birds with one stone.**  
   **Russian:** **Убить двух зайцев одним выстрелом.**  
   **Explanation:** Do two useful things at once.

4. **Spill the beans.**  
   **Russian:** **Выдать секрет / проболтаться.**  
   **Explanation:** Reveal a secret or confidential information.

5. **A blessing in disguise.**  
   **Russian:** **Не было бы счастья, да несчастье помогло.**  
   **Explanation:** Something that seems bad at first but turns out to be beneficial later.

Token usage: prompt = 87 | completion = 210 | total = 297


#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [51]:
prompt = """
Write a story about a cat in exactly three sentences.
Strictly do not use the letter e or E anywhere in the output.
Before answering, check every word and rewrite any word containing that letter.
Return only the three-sentence story.
"""
response = ask_chat(prompt, model="gpt-5.4-mini")
cat_story = response.choices[0].message.content.strip()
print(cat_story)

# Verify the constraint mechanically.
print("\nThree sentences:", len(re.findall(r"[.!?](?:\s|$)", cat_story)) == 3)
print("Contains forbidden letter:", "e" in cat_story.lower())

A sly cat ran past a moonlit yard, its paws soft on cool grass.  
It found a warm spot by a porch and sat still, watching stars glow.  
At dawn, it purrs, happy, and naps on a rug.

Three sentences: True
Contains forbidden letter: False


#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [52]:
trip_text = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."
prompt = f"""
Extract the trip information from the text below.
Return one valid JSON object only, with exactly these fields:
- destination: string, the first city mentioned
- start_date: string, date in YYYY-MM-DD format; if the year is not stated, return an empty string
- stay_duration: integer, number of nights

Text: {trip_text}
"""
json_prompt_response = ask_chat(prompt, model="gpt-5.4-mini")
print(json_prompt_response.choices[0].message.content)

{"destination":"Tokyo","start_date":"","stay_duration":5}


In [ ]:
json_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "trip_details",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "destination": {"type": "string"},
                "start_date": {"type": "string", "pattern": "^(|[0-9]{4}-[0-9]{2}-[0-9]{2})$", "description": "Date in YYYY-MM-DD format; empty string if year is not stated"},
                "stay_duration": {"type": "integer"}
            },
            "required": ["destination", "start_date", "stay_duration"],
            "additionalProperties": False
        }
    }
}
schema_response = ask_chat(
    f"Extract the trip details from this text. Do not infer a year that is not stated; use an empty string for start_date if the year is missing: {trip_text}",
    model="gpt-5.4-mini",
    response_format=json_schema
)
print(schema_response.choices[0].message.content)

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [53]:
json_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "trip_details",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "destination": {"type": "string"},
                "start_date": {"type": "string", "pattern": "^(|[0-9]{4}-[0-9]{2}-[0-9]{2})$", "description": "Date in YYYY-MM-DD format; empty string if year is not stated"},
                "stay_duration": {"type": "integer"}
            },
            "required": ["destination", "start_date", "stay_duration"],
            "additionalProperties": False
        }
    }
}
schema_response = ask_chat(
    f"Extract the trip details from this text. Do not infer a year that is not stated; use an empty string for start_date if the year is missing: {trip_text}",
    model="gpt-5.4-mini",
    response_format=json_schema
)
print(schema_response.choices[0].message.content)

{"destination":"Tokyo","start_date":"","stay_duration":5}


#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [55]:
prompt = """
Create a quiz titled **Machine learning basics quiz** with exactly five multiple-choice questions.
Follow this format exactly:
1. Question text
a) Option A
b) Option B
c) Option C
d) Option D
Mark exactly one correct option by adding a space and * at the end of that option.
Use a), b), c), d) for every question. Cover supervised learning, everyday applications, overfitting, ethical concerns, and types of machine learning.
Return only the quiz.
"""
quiz_response = ask_chat(prompt, model="gpt-5.4-mini")
print(quiz_response.choices[0].message.content)

1. Which type of machine learning uses labeled data to learn a mapping from inputs to outputs?
a) Unsupervised learning
b) Reinforcement learning
c) Supervised learning *
d) Self-directed learning

2. Which of the following is an everyday application of machine learning?
a) A calculator solving arithmetic manually
b) A music app recommending songs based on listening history *
c) A printed book changing its text automatically
d) A flashlight turning on with a switch

3. What does overfitting mean in machine learning?
a) A model performs well on training data but poorly on new data *
b) A model has too few features to learn anything
c) A model always improves as more data is added
d) A model predicts only random values

4. Which is an ethical concern in machine learning?
a) Bias in model predictions *
b) Faster internet speed
c) Increased screen brightness
d) Larger file sizes

5. Which type of machine learning learns through trial and error using rewards or penalties?
a) Supervised lear

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

####